In [ ]:
%pip install -U langchain-ollama

In [ ]:
%pip install ollama

In [ ]:
import ollama
from langchain_ollama import ChatOllama, OllamaEmbeddings

In [ ]:
LLM_MODEL = "llama3.2"
EMBEDDING_MODEL = "nomic-embed-text"

llm = ChatOllama(
    model=LLM_MODEL,
    temperature=0
)

embedding_model = OllamaEmbeddings(
    model=EMBEDDING_MODEL
)
print("Ollama LLM and embedding model initialized.")
print("LLM model:", LLM_MODEL)
print("Embedding model:", EMBEDDING_MODEL)
response = llm.invoke("Reply with exactly: Ollama is working")
print(response.content)

In [ ]:
active_model = "llama3.2"

SYSTEM_PROMPT = """
You are an expert Industrial Predictive Maintenance AI Specialist for MachineSense.

Your job is to analyze the provided machine sensor data and the Random Forest
predicted risk state, then generate a clear, actionable, concise maintenance
recommendation for plant engineers.

GUARDRAILS & RULES:

1. ONLY provide recommendations based on the sensor values and prediction
   explicitly provided in the input:
   - Vibration RMS
   - Motor Temperature
   - Average Phase Current
   - Pressure Level
   - RPM
   - Hours Since Maintenance
   - Ambient Temperature
   - Failure Probability
   - Predicted Risk State

2. NEVER invent or assume sensor readings, failure causes, machine conditions,
   or maintenance history that were not provided.

3. NEVER change, override, or contradict the provided Predicted Risk State.

4. NEVER claim that machine failure is guaranteed to occur.

5. NEVER recommend bypassing safety systems, disabling alarms, ignoring safety
   procedures, or continuing unsafe operation.

6. Do not identify a specific failed component or exact failure cause unless
   it is explicitly provided in the input.

7. If the exact cause cannot be determined from the provided data, clearly state
   that further inspection is required.

8. Keep the response strictly structured into these three sections:

Status Summary:
Immediate Recommended Action:
Safety Alert:

9. Keep the response professional, concise, precise, and practical for
   industrial maintenance engineers.
"""



In [24]:

def generate_maintenance_recommendation(row):

    user_input = f"""
Machine ID: {row['machine_id']}

Predicted Risk State: {row['risk_state']}
Failure Probability: {row['failure_probability']:.3f}

Current Sensor Data:
- Vibration RMS: {row['vibration_rms']}
- Motor Temperature: {row['temperature_motor']} °C
- Average Phase Current: {row['current_phase_avg']}
- Pressure Level: {row['pressure_level']}
- RPM: {row['rpm']}
- Hours Since Maintenance: {row['hours_since_maintenance']}
- Ambient Temperature: {row['ambient_temp']} °C

Generate the maintenance recommendation.
"""

    response = ollama.chat(
        model="llama3.2",
        messages=[
            {
                "role": "system",
                "content": SYSTEM_PROMPT
            },
            {
                "role": "user",
                "content": user_input
            }
        ]
    )

    return response["message"]["content"]